# MLP – QSAR · Morgan Fingerprint · Scaffold Split

Sieć MLP przewidująca pChEMBL (proxy aktywności biologicznej) na danych ChEMBL.
Podział na zbiory treningowy/testowy realizowany przez **Scaffold Split** (rdzeń Murcko),
analogicznie do notatnika GIN.

In [ ]:
from rdkit import Chem, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import torch.optim as optim
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score
from collections import defaultdict
import matplotlib.pyplot as plt

In [ ]:
PARQUET_PATH = "../Dane/chembl_ml_dataset_04_CHEMBL2147.parquet"
EPOCHS       = 301
BATCH_SIZE   = 256
LR           = 1e-3

## Wczytanie i filtracja danych

In [ ]:
df = pd.read_parquet(PARQUET_PATH)

# Filtracja – identyczna z notebookiem GIN
df = df[df["standard_type"] == "IC50"]
df = df[df["pchembl_value"].notna()]
df = df[df["canonical_smiles"].notna()]

target = df["target_chembl_id"].value_counts().idxmax()
df = df[df["target_chembl_id"] == target]
df = df.drop_duplicates("canonical_smiles")
df = df.reset_index(drop=True)

print("Target:  ", target)
print("Samples: ", len(df))
print("pChEMBL range:", df["pchembl_value"].min().round(2), "–", df["pchembl_value"].max().round(2))

## Morgan Fingerprint (2048 bit, radius 2)

In [ ]:
morgan = GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_fp(smiles: str) -> np.ndarray | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp  = morgan.GetFingerprint(mol)
    arr = np.zeros((2048,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

## Dataset

In [ ]:
class QSARDataset(Dataset):
    """
    Przechowuje pre-obliczone fingerprinty i wartości pChEMBL.
    Pomija cząsteczki, dla których RDKit nie może wygenerować fingerprinta.
    """
    def __init__(self, df: pd.DataFrame):
        self.fps    = []
        self.ys     = []
        self.smiles = []   # potrzebne do scaffold split

        for _, row in df.iterrows():
            fp = smiles_to_fp(row["canonical_smiles"])
            if fp is not None:
                self.fps.append(fp)
                self.ys.append(np.float32(row["pchembl_value"]))
                self.smiles.append(row["canonical_smiles"])

        self.fps = np.stack(self.fps)          # (N, 2048)
        self.ys  = np.array(self.ys)           # (N,)

    def __len__(self):
        return len(self.ys)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.fps[idx], dtype=torch.float32),
            torch.tensor(self.ys[idx],  dtype=torch.float32),
        )


dataset = QSARDataset(df)
print(f"Dataset: {len(dataset)} próbek")

## Scaffold Split

Cząsteczki są grupowane wg rdzenia Murcko. Całe grupy (scaffoldy) trafiają
albo do treningu, albo do testu – nigdy mieszane. Dzięki temu test set zawiera
**niewidziane szkielety**, co daje rzetelniejszą ocenę generalizacji niż random split.

In [ ]:
def scaffold_split(dataset: QSARDataset, frac_train: float = 0.8, seed: int = 42):
    """
    Zwraca (train_idx, test_idx) – listy indeksów do użycia z torch Subset.
    Grupy większe → priorytetowo trafiają do zbioru treningowego.
    """
    scaffold_to_idx = defaultdict(list)

    for idx, smi in enumerate(dataset.smiles):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            scaffold = ""
        else:
            try:
                scaffold = MurckoScaffold.MurckoScaffoldSmiles(
                    mol=mol, includeChirality=False
                )
            except Exception:
                scaffold = ""
        scaffold_to_idx[scaffold].append(idx)

    # Malejąco wg rozmiaru grupy
    groups = sorted(
        scaffold_to_idx.values(),
        key=lambda g: (len(g), g[0]),
        reverse=True,
    )

    cutoff = int(frac_train * len(dataset))
    train_idx, test_idx = [], []

    for group in groups:
        if len(train_idx) < cutoff:
            train_idx.extend(group)
        else:
            test_idx.extend(group)

    print(f"Scaffold split → train: {len(train_idx)}, test: {len(test_idx)}")
    print(f"Unikalnych scaffoldów: {len(scaffold_to_idx)}")
    return train_idx, test_idx


train_idx, test_idx = scaffold_split(dataset, frac_train=0.8)

train_ds = Subset(dataset, train_idx)
test_ds  = Subset(dataset, test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## Model MLP

Architektura z BatchNorm między warstwami – stabilizuje trening
i zmniejsza wrażliwość na learning rate.

In [ ]:
def build_mlp(dropout: float = 0.3) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(2048, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        nn.Dropout(dropout),

        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(dropout),

        nn.Linear(512, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(dropout),

        nn.Linear(128, 1),
    )

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = build_mlp(dropout=0.3).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nParametry: {n_params:,}  |  Device: {device}")

## Learning Curve (przed pełnym treningiem)

Sprawdzamy jak model skaluje się z ilością danych treningowych.
Każda frakcja trenuje **ten sam model od zera** przez 30 epok.

In [ ]:
fractions    = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
lc_val_r2    = []
lc_train_r2  = []
LC_EPOCHS    = 30

for frac in fractions:
    size   = max(1, int(len(train_idx) * frac))
    subset = Subset(dataset, train_idx[:size])
    loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True)

    lc_model = build_mlp().to(device)
    lc_opt   = optim.Adam(lc_model.parameters(), lr=LR)

    for _ in range(LC_EPOCHS):
        lc_model.train()
        for X, y in loader:
            X, y = X.to(device), y.to(device).view(-1, 1)
            lc_opt.zero_grad()
            criterion(lc_model(X), y).backward()
            lc_opt.step()

    lc_model.eval()

    def _eval(ldr):
        ps, ts = [], []
        with torch.no_grad():
            for X, y in ldr:
                ps.extend(lc_model(X.to(device)).cpu().numpy().flatten())
                ts.extend(y.numpy())
        return r2_score(ts, ps)

    lc_train_r2.append(_eval(DataLoader(subset, batch_size=BATCH_SIZE)))
    lc_val_r2.append(_eval(test_loader))
    print(f"  frac={frac:.1f}  train_R²={lc_train_r2[-1]:.3f}  val_R²={lc_val_r2[-1]:.3f}")

plt.figure(figsize=(7, 4))
plt.plot(fractions, lc_train_r2, 'o-', label="Train R²")
plt.plot(fractions, lc_val_r2,   's--', label="Val R²")
plt.xlabel("Frakcja zbioru treningowego")
plt.ylabel("R²")
plt.title("Learning Curve (MLP, Scaffold split)")
plt.legend()
plt.tight_layout()
plt.show()

## Trening właściwy

In [ ]:
# Zresetuj model i optymalizator przed pełnym treningiem
model     = build_mlp(dropout=0.3).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)

train_losses = []
val_losses   = []

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    total = 0.0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device).view(-1, 1)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        total += loss.item()
    train_losses.append(total / len(train_loader))

    # --- Eval ---
    model.eval()
    vtotal = 0.0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device).view(-1, 1)
            vtotal += criterion(model(X), y).item()
    val_losses.append(vtotal / len(test_loader))

    if epoch % 50 == 0:
        print(f"Epoch {epoch:>4}: train={train_losses[-1]:.4f}  val={val_losses[-1]:.4f}")

    if epoch % 100 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, f'../Dane/Modele/mlp_scaffold_checkpoint_{epoch}.pt')

In [ ]:
model.eval()
preds, targets = [], []

with torch.no_grad():
    for X, y in test_loader:
        preds.extend(model(X.to(device)).cpu().numpy().flatten())
        targets.extend(y.numpy())

preds   = np.array(preds)
targets = np.array(targets)

r2   = r2_score(targets, preds)
rmse = np.sqrt(mean_squared_error(targets, preds))

threshold   = 6.0
accuracy    = accuracy_score((targets >= threshold).astype(int),
                             (preds   >= threshold).astype(int))

print(f"R²       : {r2:.4f}")
print(f"RMSE     : {rmse:.4f}")
print(f"Accuracy : {accuracy:.4f}  (próg pChEMBL={threshold})")

## Wyniki

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label="Train loss")
axes[0].plot(val_losses,   label="Validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Training curves (MLP + Scaffold split)")
axes[0].legend()

mn, mx = targets.min(), targets.max()
axes[1].scatter(targets, preds, alpha=0.4, s=15)
axes[1].plot([mn, mx], [mn, mx], 'r--', label='ideal')
axes[1].set_xlabel("True pChEMBL")
axes[1].set_ylabel("Predicted pChEMBL")
axes[1].set_title(f"R²={r2:.3f}  RMSE={rmse:.3f}")
axes[1].legend()

plt.tight_layout()
plt.show()

## Dotrenowanie z LR Scheduler

In [ ]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=30
)

START_EPOCH = EPOCHS
END_EPOCH   = EPOCHS + 700

train_losses_cont = []
val_losses_cont   = []

for epoch in range(START_EPOCH, END_EPOCH):
    model.train()
    total = 0.0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device).view(-1, 1)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        total += loss.item()
    train_losses_cont.append(total / len(train_loader))

    model.eval()
    vtotal = 0.0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device).view(-1, 1)
            vtotal += criterion(model(X), y).item()
    val_losses_cont.append(vtotal / len(test_loader))

    scheduler.step(val_losses_cont[-1])
    lr_now = optimizer.param_groups[0]['lr']

    if epoch % 50 == 0:
        print(f"Epoch {epoch:>5}: train={train_losses_cont[-1]:.4f}  "
              f"val={val_losses_cont[-1]:.4f}  lr={lr_now:.6f}")

    if epoch % 100 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, f'../Dane/Modele/mlp_scaffold_checkpoint_{epoch}.pt')

In [ ]:
model.eval()
preds, targets = [], []

with torch.no_grad():
    for X, y in test_loader:
        preds.extend(model(X.to(device)).cpu().numpy().flatten())
        targets.extend(y.numpy())

preds   = np.array(preds)
targets = np.array(targets)

r2   = r2_score(targets, preds)
rmse = np.sqrt(mean_squared_error(targets, preds))
acc  = accuracy_score((targets >= 6.0).astype(int), (preds >= 6.0).astype(int))

print(f"R²       : {r2:.4f}")
print(f"RMSE     : {rmse:.4f}")
print(f"Accuracy : {acc:.4f}")

In [ ]:
epoki = range(START_EPOCH, START_EPOCH + len(train_losses_cont))
plt.figure(figsize=(10, 4))
plt.plot(epoki, train_losses_cont, label="Train loss")
plt.plot(epoki, val_losses_cont,   label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Dotrenowanie MLP z ReduceLROnPlateau")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
torch.save(model.state_dict(), "../Dane/mlp_scaffold_01.pt")